# Delegate to another agent

This notebook deploys `research-coordinator`, an agent that delegates web research to `web-researcher`. See the [README](./README.md) for the same steps in a terminal.

kagent and MCP servers must be enabled on your prokube cluster.

## Setup

Set up this repository's kagent helpers from [kagent_helpers.py](../shared/kagent_helpers.py). Adding their directory to `sys.path` lets the notebook import them without installing anything.

In [ ]:
import sys

sys.path.insert(0, "../shared")  # kagent helpers (shared by all kagent examples)

import kagent_helpers

In [ ]:
import os
import subprocess
from getpass import getpass
from pathlib import Path

with open("/var/run/secrets/kubernetes.io/serviceaccount/namespace") as f:
    namespace = f.read().strip()


def kubectl_apply(manifest: str) -> None:
    subprocess.run(["kubectl", "apply", "-f", "-"], input=manifest, text=True, check=True)

## Prerequisites

The coordinator needs `web-researcher` from [`agent-with-mcp`](../agent-with-mcp/) and everything it depends on. The next cells create whatever is missing; `kubectl apply` leaves existing resources unchanged.

The agents call Claude with the key in the `anthropic-api-key` Secret. The next cell creates the Secret unless it already exists, for example because you created it on the **Secrets** page of the prokube UI. It prompts for the key unless `ANTHROPIC_API_KEY` is set.

In [ ]:
secret_exists = subprocess.run(
    ["kubectl", "get", "secret", "anthropic-api-key"], capture_output=True
).returncode == 0
if not secret_exists:
    anthropic_api_key = os.environ.get("ANTHROPIC_API_KEY") or getpass("Anthropic API key: ")
    kubectl_apply(
        Path("../basic-agent/anthropic-secret.yaml")
        .read_text()
        .replace("<your-anthropic-api-key>", anthropic_api_key)
    )

In [ ]:
!kubectl apply -f ../basic-agent/anthropic-model.yaml
!kubectl wait --for=condition=Accepted modelconfigs.kagent.dev/anthropic-haiku --timeout=2m
!kubectl apply -f ../../mcp-servers/deploy-upstream-mcp-server/fetch-server.yaml
!kubectl wait --for=condition=Ready mcpservers.toolhive.stacklok.dev/fetch-example --timeout=3m

In [ ]:
fetch_tool = kagent_helpers.gateway_mcp_tool_name("fetch-example", "fetch")
kubectl_apply(
    Path("../agent-with-mcp/web-researcher.yaml").read_text().replace("- fetch\n", f"- {fetch_tool}\n")
)
!kubectl wait --for=condition=Ready agents.kagent.dev/web-researcher --timeout=3m

## Deploy

In [ ]:
!kubectl apply -f research-coordinator.yaml
!kubectl wait --for=condition=Ready agents.kagent.dev/research-coordinator --timeout=3m

## Talk to the agent from this Lab

Labs reach agents through the internal Agent Gateway route without an API key. You can also open **Agents** in the prokube UI, select `research-coordinator`, and chat there.

Expect one sentence about what prokube offers that includes the source URL `https://prokube.ai`, as returned by `web-researcher`.

In [ ]:
internal_url = kagent_helpers.workspace_a2a_url(namespace, "research-coordinator")
print(internal_url)

In [ ]:
prompt = "Research https://prokube.ai and summarize what prokube offers in one sentence. Include the source URL."
print(kagent_helpers.send_message(internal_url, prompt))

## Connect with an API key

Clients outside the cluster use the agent's external route, `https://<your-prokube-domain>/svc/a2a/<workspace>/research-coordinator`, with an API key. See [Public vs. Internal Traffic](https://prokube.ai/docs/1.9/platform/agent_gateway.html#public-vs-internal-traffic) for how this route differs from the internal one.

Enter your prokube domain, the host name you open prokube at, for example `my-cluster.prokube.cloud`. The cell uses `PROKUBE_DOMAIN` instead if it is set:

In [ ]:
domain = os.environ.get("PROKUBE_DOMAIN") or input("prokube domain (e.g. my-cluster.prokube.cloud): ")

Create a key with access to `research-coordinator` on the [API Keys page](/pkui/ai-gateway/keys) (`https://<your-prokube-domain>/pkui/ai-gateway/keys`), then paste it when prompted below. The cell uses `A2A_API_KEY` instead if it is set:

In [ ]:
api_key = os.environ.get("A2A_API_KEY") or getpass("Agent API key: ")

Send the same prompt through the agent's external URL:

In [ ]:
api_key_url = f"https://{domain}/svc/a2a/{namespace}/research-coordinator"
print(api_key_url)

In [ ]:
print(kagent_helpers.send_message(api_key_url, prompt, api_key=api_key))

## Clean up

Uncomment the command and run the cell to delete `research-coordinator`. This leaves `web-researcher` and its prerequisites. To remove everything, also run the clean-up cells of the earlier notebooks and delete the Fetch server.

In [ ]:
# !kubectl delete -f research-coordinator.yaml